# FranchiseOps AI RAG Knowledge Base Builder
This notebook scrapes and curates a knowledge base for the FranchiseOps RAG system.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

!pip install -q langchain langchain-community langchain-text-splitters -U langchain-core sentence-transformers faiss-cpu pymupdf beautifulsoup4 requests==2.32.4 urllib3 tqdm vaderSentiment textblob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

import os
import json
import time
import requests
import fitz  # PyMuPDF
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import urllib3
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

RAG_DIR = '/content/drive/MyDrive/FranchiseOps_AI/rag_documents'
os.makedirs(RAG_DIR, exist_ok=True)
print(f'✅ Output Directory Ready: {RAG_DIR}')


Mounted at /content/drive
✅ Output Directory Ready: /content/drive/MyDrive/FranchiseOps_AI/rag_documents


In [ ]:
HTML_SOURCES = [
    # Marketing & Consumer Research
    "https://www.marketingweek.com",
    "https://hbr.org/topic/subject/marketing",
    "https://www.nielsen.com/insights",
    "https://www.mckinsey.com/capabilities/growth-marketing-and-sales/our-insights",
    "https://www.thinkwithgoogle.com",
    "https://www.campaignlive.co.uk",
    "https://www.warc.com/newsandopinion/opinion",

    # Customer Experience
    "https://www.pwc.com/us/en/services/consulting/library/consumer-intelligence-series.html",
    "https://www.zendesk.com/blog/customer-experience",
    "https://www.salesforce.com/resources/articles/customer-experience",

    # HR & Attrition Research
    "https://www.shrm.org/topics-tools/tools/how-to-guides/how-to-conduct-stay-interviews",
    "https://www.gallup.com/workplace/247391/fixable-problem-costs-businesses-trillion.aspx",
    "https://hbr.org/topic/subject/hr-management",
    "https://www.mckinsey.com/capabilities/people-and-organizational-performance/our-insights",

    # Food Safety & FSSAI
    "https://www.fssai.gov.in",
    "https://www.fssai.gov.in/cms/food-safety-and-standards-act-2006.php",
    "https://www.fssai.gov.in/cms/rules.php",
    "https://www.fssai.gov.in/cms/regulations.php",
    "https://www.fssai.gov.in/cms/gazette-notifications.php",
    "https://www.fssai.gov.in/cms/licensing.php",

    # Labour Laws
    "https://labour.gov.in/minimum-wages-act",
    "https://labour.gov.in/payment-of-wages-act",
    "https://labour.gov.in/maternity-benefit-act",
    "https://labour.gov.in/child-labour",
    "https://labour.gov.in/factories-act",
    "https://labour.gov.in/employees-provident-fund-organisation",
    "https://labour.gov.in/employees-state-insurance-corporation",
    "https://labour.gov.in/occupational-safety-and-health",
    "https://labour.gov.in/social-security",
    "https://labour.gov.in/industrial-relations",
    "https://labour.gov.in/bonus-act",
    "https://labour.gov.in/gratuity-act",

    # OSHA
    "https://www.osha.gov/workers",
    "https://www.osha.gov/employers",
    "https://www.osha.gov/laws-regs",
    "https://www.osha.gov/heat-exposure",
    "https://www.osha.gov/young-workers",
    "https://www.osha.gov/ergonomics",
    "https://www.osha.gov/personal-protective-equipment",

    # FDA
    "https://www.fda.gov/food/guidance-regulation-food-and-dietary-supplements",
    "https://www.fda.gov/food/buy-store-serve-safe-food",
    "https://www.fda.gov/food/new-era-smarter-food-safety",
    "https://www.fda.gov/food/food-labeling-nutrition",

    # WHO & International
    "https://www.who.int/news-room/fact-sheets/detail/food-safety",
    "https://www.who.int/health-topics/food-safety",
    "https://www.codexalimentarius.org",
    "https://www.fao.org/food-safety/en",
    "https://efsa.europa.eu/en/topics/topic/food-safety",
    "https://www.epfindia.gov.in",
    "https://www.esic.gov.in",
    "https://www.bis.gov.in",
    "https://apeda.gov.in",
    "https://www.mofpi.gov.in",
    "https://niti.gov.in",
    "https://www.startupindia.gov.in",
    "https://www.msme.gov.in",
    "https://mca.gov.in",
    "https://consumerhelpline.gov.in",
    "https://www.ncdrc.nic.in",
    "https://www.ilo.org/global/topics/safety-and-health-at-work/lang--en/index.htm",
    "https://www.ilo.org/global/topics/wages/minimum-wages/lang--en/index.htm",
    "https://hbr.org/topic/subject/hr-management",
    "https://www.shrm.org/topics-tools/tools/how-to-guides/how-to-develop-employee-handbook",
    "https://www.foodsafety.gov",
    "https://www.food.gov.uk",
    "https://www.food.gov.uk/business-guidance",
    "https://www.foodstandards.gov.au",
    "https://www.canada.ca/en/health-canada/services/food-nutrition.html",
    "https://www.sqfi.com",
    "https://www.brcgs.com/our-standards/food-safety",
    "https://www.mygfsi.com",
    "https://www.iso.org/committee/47638.html",
    "https://www.fao.org/nutrition/en",
    "https://www.wcfc.co",
    "https://www.bfa.org.uk",
    "https://www.ftc.gov/tips-advice/business-center/guidance/franchise-rule",
    "https://www.sba.gov/business-guide/launch-your-business/buy-franchise",
    "https://www.ifa.com",
    "https://www.qsrmagazine.com",
    "https://www.nrn.com",
    "https://www.restaurant.org/research-and-media/research/research-reports/state-of-the-industry",
    "https://www.mckinsey.com/capabilities/people-and-organizational-performance/our-insights",
    "https://www.gallup.com/workplace/247391/fixable-problem-costs-businesses-trillion.aspx",
    "https://www.nielsen.com/insights",
    "https://www.pwc.com/us/en/services/consulting/library/consumer-intelligence-series.html",
    "https://www.zendesk.com/blog/customer-experience",
    "https://www.salesforce.com/resources/articles/customer-experience",
    "https://www.indianspices.com",
    "https://www.coffeeboard.gov.in",
    "https://www.teaboard.gov.in",
    "https://mpeda.gov.in",
    "https://www.india.gov.in/spotlight/food-processing",
    "https://efsa.europa.eu/en/topics/topic/food-safety",
    "https://www.fao.org/food-safety/en",
    "https://www.who.int/teams/nutrition-and-food-safety/food-safety",
    "https://www.thinkwithgoogle.com",
    "https://www.marketingweek.com",
    "https://hbr.org/topic/subject/marketing"
]

PDF_SOURCES = [
    "https://www.fssai.gov.in/upload/uploadfiles/files/Food_Safety_and_Standards_Act_2006.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Licensing_and_Registration_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Packaging_and_Labelling_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Food_Product_Standards_and_Food_Additives_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Prohibition_and_Restriction_on_Sales_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Contaminants_Toxins_and_Residues_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Laboratory_and_Sample_Analysis_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Food_Safety_and_Standards_Rules_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Compendium_Food_Safety_Standards_Act.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/44633/9789241501651_eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/255027/9789241512442-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/43038/9241546123_eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/42913/9241546468.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/326765/9789240004467-eng.pdf",
    "https://www.fao.org/3/a0512e/a0512e00.pdf",
    "https://www.fao.org/3/i3794e/i3794e.pdf",
    "https://www.fao.org/3/y1579e/y1579e00.pdf",
    "https://www.fao.org/3/w9raw-e.pdf",
    "https://www.fao.org/3/y1390e/y1390e00.pdf",
    "https://www.fao.org/3/a-i4955e.pdf",
    "https://www.fao.org/3/cb4474en/cb4474en.pdf",
    "https://www.fao.org/3/ca5399en/ca5399en.pdf",
    "https://www.fao.org/3/i0142e/i0142e.pdf",
    "https://www.fao.org/3/y4893e/y4893e.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_norm/---normes/documents/publication/wcms_087817.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---dgreports/---dcomm/documents/publication/wcms_067588.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_protect/---protrav/---travail/documents/publication/wcms_712957.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_emp/---emp_ent/documents/publication/wcms_093580.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---asia/---ro-bangkok/---sro-new_delhi/documents/publication/wcms_631470.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3165.pdf",
    "https://www.osha.gov/sites/default/files/publications/3148-06R-2011-English.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3151.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha2254.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3170.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3071.pdf",
    "https://www.osha.gov/sites/default/files/publications/OSHA_FS-3696.pdf",
    "https://www.ftc.gov/sites/default/files/documents/plain-language/bus70-franchise-rule.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/36613/9781464816109.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/35016/9781464816123.pdf",
    "https://labour.gov.in/sites/default/files/THE_MINIMUM_WAGES_ACT_1948.pdf",
    "https://labour.gov.in/sites/default/files/PaymentofWagesAct1936_0.pdf",
    "https://labour.gov.in/sites/default/files/TheMaternityBenefitAct_1961.pdf",
    "https://labour.gov.in/sites/default/files/payment_of_gratuity_act.pdf",
    "https://labour.gov.in/sites/default/files/ThePaymentofBonusAct1965.pdf",
    "https://www.fao.org/fao-who-codexalimentarius/sh-proxy/en/?lnk=1&url=https%3A%2F%2Fworkspace.fao.org%2Fsites%2Fcodex%2FStandards%2FCXP+1-1969%2FCXP001e.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Document_Organic_Foods.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Note_Nutraceuticals.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Milk_and_Milk_Products.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Meat_and_Meat_Products.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Fruits_and_Vegetables.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Document_Health_Supplements.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Note_Food_Additives.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Document_Edible_Oils.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/FSS_Organic_Foods_Regulations_2017.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Draft_FSS_Food_Recall_Procedure_Regulations_2017.pdf",
    "https://consumeronline.gov.in/documents/ConsumerProtection-Act-2019.pdf",
    "https://www.epfindia.gov.in/site_docs/PDFs/Circulars/Y2022-23/Circular_EPFO_0111_2022.pdf",
    "https://www.sba.gov/sites/default/files/2022-08/Franchise-Guide.pdf",
    "https://www.mckinsey.com/~/media/McKinsey/Business%20Functions/People%20and%20Organizational%20Performance/Our%20Insights/Reinventing%20the%20organization/Reinventing-the-organization.pdf",
    "https://www.foodstandards.gov.au/sites/default/files/documents/Meat%20Industry%20Operational%20Audit%20Guide.pdf",
    "https://www.ifi.unc.edu/wp-content/uploads/sites/863/2019/12/IFI-Report-Food-safety-culture.pdf",
    "https://efsa.onlinelibrary.wiley.com/doi/epdf/10.2903/j.efsa.2020.6098",
    "https://www.shrm.org/hr-today/trends-and-forecasting/research-and-surveys/Documents/SHRM%20Employee%20Job%20Satisfaction%20and%20Engagement.pdf",
    "https://www.gallup.com/workplace/349484/state-of-the-global-workplace-2022-report.aspx",
    "https://www.fssai.gov.in/upload/uploadfiles/files/FOSTAC_Training_Module_Basic.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/FOSTAC_Training_Module_Special.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Hygiene_Rating_Scheme_Guidelines.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Clean_Street_Food_Hub_Guidelines.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---asia/---ro-bangkok/---sro-new_delhi/documents/publication/wcms_766448.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_protect/---protrav/---safework/documents/instructionalmaterial/wcms_113522.pdf",
    "https://bis.gov.in/wp-content/uploads/2022/02/Annual-Report-2020-21.pdf",
    "https://apeda.gov.in/apedawebsite/ANNUAL_REPORT/APEDA-Annual-Report-2021-22.pdf",
    "https://www.msme.gov.in/sites/default/files/MSME-Annual-Report-2021-22.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/344474/9789240030060-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/274671/9789241514217-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/204348/9789241510066_eng.pdf",
    "https://www.fao.org/3/ca0640en/CA0640EN.pdf",
    "https://www.fao.org/3/i9933en/i9933en.pdf",
    "https://www.fao.org/3/cc0461en/cc0461en.pdf",
    "https://www.fao.org/3/cb7408en/cb7408en.pdf",
    "https://niti.gov.in/sites/default/files/2022-12/Food-Processing-Report.pdf",
    "https://www.mofpi.gov.in/sites/default/files/annual_report_2020-21.pdf",
    "https://www.restaurant.org/downloads/pdfs/research/whats_hot_2023.pdf",
    "https://www.osha.gov/sites/default/files/publications/OSHA3604.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3180.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha_3590.pdf",
    "https://www.fao.org/fao-who-codexalimentarius/sh-proxy/en/?lnk=1&url=https%3A%2F%2Fworkspace.fao.org%2Fsites%2Fcodex%2FStandards%2FCXS+1-1985%2FCXS001e.pdf",
    "https://www.fao.org/fao-who-codexalimentarius/sh-proxy/en/?lnk=1&url=https%3A%2F%2Fworkspace.fao.org%2Fsites%2Fcodex%2FStandards%2FCXS+193-1995%2FCXS193e.pdf"
]

print(f"HTML Sources: {len(HTML_SOURCES)}")
print(f"PDF Sources: {len(PDF_SOURCES)}")

HTML Sources: 98
PDF Sources: 88


In [ ]:
import os, json, time, requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import fitz  # PyMuPDF
from tqdm import tqdm

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

manifest_path = os.path.join(RAG_DIR, 'manifest.json')
manifest = {}
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)

def save_manifest():
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

def get_with_retry(url, max_retries=3, timeout=20):
    """GET request with exponential backoff and SSL fallback."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout, verify=True)
            resp.raise_for_status()
            return resp
        except requests.exceptions.SSLError:
            try:
                resp = requests.get(url, headers=HEADERS, timeout=timeout, verify=False)
                resp.raise_for_status()
                return resp
            except Exception as e:
                if attempt == max_retries - 1:
                    raise e
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            wait = 2 ** attempt
            time.sleep(wait)
    return None

def harvest_pdfs_from_page(url, base_domain=None):
    """
    Visit a webpage and auto-discover all PDF links embedded inside it.
    Returns a list of absolute PDF URLs found on the page.
    """
    discovered = []
    try:
        resp = get_with_retry(url, timeout=15)
        if resp is None:
            return discovered
        soup = BeautifulSoup(resp.content, 'html.parser')
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href'].strip()
            # Check if link ends with .pdf or contains /pdf/ in path
            if href.lower().endswith('.pdf') or '/pdf/' in href.lower():
                # Convert relative URLs to absolute
                if href.startswith('http'):
                    abs_url = href
                elif href.startswith('//'):
                    abs_url = 'https:' + href
                elif href.startswith('/'):
                    parsed = urlparse(url)
                    abs_url = f"{parsed.scheme}://{parsed.netloc}{href}"
                else:
                    abs_url = urljoin(url, href)
                # Clean URL (remove fragments)
                abs_url = abs_url.split('#')[0]
                if abs_url not in discovered:
                    discovered.append(abs_url)
    except Exception as e:
        print(f"  ⚠️  PDF harvest failed for {url}: {e}")
    return discovered

def scrape_html(url):
    """Scrape HTML page text and save as .txt file."""
    if manifest.get(url) == 'success':
        return 'skipped'
    try:
        resp = get_with_retry(url)
        if resp is None:
            manifest[url] = 'failed: no response'
            return 'failed'
        soup = BeautifulSoup(resp.content, 'html.parser')
        # Remove script and style elements
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()
        text = soup.get_text(separator=' ', strip=True)
        if len(text.strip()) < 100:
            manifest[url] = 'failed: too short'
            return 'failed'
        safe_name = url.replace('https://', '').replace('http://', '').replace('/', '_').replace('?', '_')[:80]
        out_path = os.path.join(RAG_DIR, f'html_{safe_name}.txt')
        with open(out_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(f'SOURCE: {url}\n\n{text}')
        manifest[url] = 'success'
        save_manifest()
        return 'success'
    except Exception as e:
        manifest[url] = f'failed: {str(e)[:80]}'
        save_manifest()
        return 'failed'

def scrape_pdf(url):
    """Download a PDF and extract all text pages."""
    if manifest.get(url) == 'success':
        return 'skipped'
    try:
        resp = get_with_retry(url, timeout=45)
        if resp is None:
            manifest[url] = 'failed: no response'
            return 'failed'
        # Check content type
        content_type = resp.headers.get('Content-Type', '')
        if 'pdf' not in content_type.lower() and not url.lower().endswith('.pdf'):
            manifest[url] = 'failed: not a pdf'
            return 'failed'
        doc = fitz.open(stream=resp.content, filetype='pdf')
        text = '\n'.join([page.get_text() for page in doc])
        doc.close()
        if len(text.strip()) < 50:
            manifest[url] = 'failed: empty pdf (scanned image)'
            return 'failed'
        safe_name = url.replace('https://', '').replace('http://', '').replace('/', '_').replace('?', '_')[:80]
        out_path = os.path.join(RAG_DIR, f'pdf_{safe_name}.txt')
        with open(out_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(f'SOURCE: {url}\n\n{text}')
        manifest[url] = 'success'
        save_manifest()
        return 'success'
    except Exception as e:
        manifest[url] = f'failed: {str(e)[:80]}'
        save_manifest()
        return 'failed'

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: Scrape all HTML pages AND auto-harvest PDF links from each page
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 60)
print('PHASE 1: Scraping HTML pages + Auto-harvesting embedded PDFs')
print('=' * 60)

discovered_pdfs = set()
html_stats = {'success': 0, 'skipped': 0, 'failed': 0}

for url in tqdm(HTML_SOURCES, desc='🌐 HTML Pages'):
    result = scrape_html(url)
    html_stats[result] = html_stats.get(result, 0) + 1

    # Auto-harvest PDF links from every HTML page
    if result in ('success', 'skipped'):
        pdfs_found = harvest_pdfs_from_page(url)
        for pdf_url in pdfs_found:
            discovered_pdfs.add(pdf_url)
        if pdfs_found:
            print(f'  📎 Found {len(pdfs_found)} PDFs on {url[:60]}')
    time.sleep(1)  # Polite delay

print(f'\n✅ HTML done: {html_stats}')
print(f'📎 Auto-discovered {len(discovered_pdfs)} PDFs from HTML pages')

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: Merge discovered PDFs with static PDF_SOURCES list
# ─────────────────────────────────────────────────────────────────────────────
all_pdf_urls = list(set(PDF_SOURCES) | discovered_pdfs)
print(f'\n📚 Total unique PDFs to process: {len(all_pdf_urls)}')
print(f'  → From static list: {len(PDF_SOURCES)}')
print(f'  → Auto-discovered:  {len(discovered_pdfs)}')

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: Download & extract all PDFs
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('PHASE 3: Downloading & Extracting All PDFs')
print('=' * 60)

pdf_stats = {'success': 0, 'skipped': 0, 'failed': 0}
for url in tqdm(all_pdf_urls, desc='📄 PDFs'):
    result = scrape_pdf(url)
    pdf_stats[result] = pdf_stats.get(result, 0) + 1
    if result == 'success':
        time.sleep(0.5)  # Polite delay for successful downloads

print(f'\n✅ PDF done: {pdf_stats}')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
txt_files = [f for f in os.listdir(RAG_DIR) if f.endswith('.txt')]
print('\n' + '=' * 60)
print('📊 SCRAPING COMPLETE — SUMMARY')
print('=' * 60)
print(f'  HTML pages scraped:      {html_stats["success"]} success, {html_stats["skipped"]} skipped, {html_stats["failed"]} failed')
print(f'  PDFs auto-discovered:    {len(discovered_pdfs)}')
print(f'  PDFs downloaded:         {pdf_stats["success"]} success, {pdf_stats["skipped"]} skipped, {pdf_stats["failed"]} failed')
print(f'  Total .txt files in RAG: {len(txt_files)}')
print(f'  Manifest entries:        {len(manifest)}')
print('=' * 60)


PHASE 1: Scraping HTML pages + Auto-harvesting embedded PDFs


🌐 HTML Pages:  33%|███▎      | 32/98 [03:36<02:33,  2.32s/it]

  📎 Found 9 PDFs on https://www.osha.gov/workers


🌐 HTML Pages:  34%|███▎      | 33/98 [03:37<02:10,  2.00s/it]

  📎 Found 5 PDFs on https://www.osha.gov/employers


...

MuPDF error: format error: object is not a stream



📄 PDFs:  85%|████████▍ | 434/512 [1:11:27<06:17,  4.84s/it]

MuPDF error: format error: object is not a stream



📄 PDFs: 100%|██████████| 512/512 [1:27:44<00:00, 10.28s/it]


✅ PDF done: {'success': 265, 'skipped': 0, 'failed': 247}

📊 SCRAPING COMPLETE — SUMMARY
  HTML pages scraped:      43 success, 8 skipped, 47 failed
  PDFs auto-discovered:    425
  PDFs downloaded:         265 success, 0 skipped, 247 failed
  Total .txt files in RAG: 344
  Manifest entries:        599


In [ ]:
print(f"\nLoading {len(txt_files)} scraped text files from Drive...")
documents = []
for fname in tqdm(txt_files, desc="📂 Loading Docs"):
    filepath = os.path.join(RAG_DIR, fname)
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    if len(text.strip()) > 50:
        documents.append(Document(page_content=text, metadata={"source": fname, "type": "scraped"}))

print(f"\n✅ Loaded {len(documents)} documents into memory.")



Loading 344 scraped text files from Drive...


📂 Loading Docs: 100%|██████████| 344/344 [00:04<00:00, 80.96it/s] 


✅ Loaded 344 documents into memory.


In [ ]:
curated_sops = [
    {"id": "KB-101", "content": "Minimum freezer temperature must be maintained at -18°C or below at all times."},
    {"id": "KB-102", "content": "Store closing procedures include counting the cash drawer, securing the safe, turning off non-essential equipment, and setting the alarm."},
    {"id": "KB-103", "content": "Staff must wash hands with soap and warm water for at least 20 seconds before starting a shift, after using the restroom, and after handling raw meat."},
    {"id": "KB-104", "content": "Food preparation surfaces must be sanitized every 2 hours using the approved quaternary ammonium sanitizer solution."},
    {"id": "KB-105", "content": "The FIFO (First In, First Out) method must be strictly followed for all perishable inventory."},
    {"id": "KB-106", "content": "Daily temperature logs for all refrigeration units must be recorded at 8:00 AM, 2:00 PM, and 8:00 PM."},
    {"id": "KB-107", "content": "Customer complaints regarding food quality must be immediately escalated to the Shift Manager for resolution."},
    {"id": "KB-108", "content": "Spills on the customer floor must be marked with a wet floor sign and cleaned up within 3 minutes."},
    {"id": "KB-109", "content": "All staff members must wear the complete, approved uniform including name tag, hat/visor, and slip-resistant shoes."},
    {"id": "KB-110", "content": "Waste bins must be emptied when they are 3/4 full; never allow trash to overflow."},
    {"id": "KB-111", "content": "A minimum of 3 staff members (1 Manager, 1 Front-of-House, 1 Back-of-House) are required per shift."},
    {"id": "KB-112", "content": "FSSAI license must be prominently displayed near the point of sale at all times."},
    {"id": "KB-113", "content": "Penalties for FSSAI non-compliance can range from warning letters to license suspension and fines up to ₹2,00,000 depending on the severity."},
    {"id": "KB-114", "content": "Pest control services must be scheduled monthly, and inspection reports must be kept in the compliance binder."},
    {"id": "KB-115", "content": "Only approved vendors may be used for sourcing raw ingredients and packaging materials."},
    {"id": "KB-116", "content": "Deep fryers must be filtered daily and the oil completely changed every 3 days or when it fails the color check test."},
    {"id": "KB-117", "content": "Fire extinguishers must be inspected monthly by the Manager and annually by a certified professional."},
    {"id": "KB-118", "content": "All new employees must complete the 40-hour basic operational training program before working independently."},
    {"id": "KB-119", "content": "Cash drops to the safe must be performed whenever the register drawer exceeds ₹20,000."},
    {"id": "KB-120", "content": "The store key must never be duplicated, and must be returned immediately upon termination of employment."},
    {"id": "KB-121", "content": "Marketing ROI minimum threshold: >15% ROI is required for any marketing campaign renewal."},
    {"id": "KB-122", "content": "Customer complaint categories and resolution SLA per category: Critical (2 hours), High (24 hours), Medium (48 hours), Low (72 hours)."},
    {"id": "KB-123", "content": "Staff performance review frequency is quarterly, utilizing a 5-point scoring methodology based on attendance, customer service, and task completion."},
    {"id": "KB-124", "content": "Social media compliance requires a response within 2 hours for all negative reviews on major platforms."},
    {"id": "KB-125", "content": "Outlet opening checklist includes a 30-point pre-launch audit covering equipment, inventory, staff readiness, and regulatory compliance."}
]

with open('kb_franchise.json', 'w') as f:
    json.dump(curated_sops, f, indent=4)

for sop in curated_sops:
    documents.append(Document(page_content=sop["content"], metadata={"source": "curated_sop", "id": sop["id"]}))

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local("franchiseops_faiss_index")
print("FAISS index built and saved successfully.")

/tmp/ipykernel_636/3792370220.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built and saved successfully.


In [ ]:
print("\n--- DRY RUN: 7 Test Queries ---")
test_queries = [
    "What is the minimum freezer temperature?",
    "How many staff are required per shift?",
    "What is the handwashing procedure?",
    "What are the penalties for FSSAI non-compliance?",
    "How should customer complaints be escalated?",
    "What is the minimum marketing ROI threshold for campaign renewal?",
    "What are the staff performance review requirements?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    docs = vectorstore.similarity_search(query, k=1)
    if docs:
        print(f"Answer (Snippet): {docs[0].page_content}")
        print(f"Source: {docs[0].metadata.get('source', 'Unknown')} | ID: {docs[0].metadata.get('id', 'N/A')}")
    else:
        print("No relevant documents found.")


--- DRY RUN: 7 Test Queries ---

Query: What is the minimum freezer temperature?
Answer (Snippet): Minimum freezer temperature must be maintained at -18°C or below at all times.
Source: curated_sop | ID: KB-101

Query: How many staff are required per shift?
Answer (Snippet): A minimum of 3 staff members (1 Manager, 1 Front-of-House, 1 Back-of-House) are required per shift.
Source: curated_sop | ID: KB-111

Query: What is the handwashing procedure?
Answer (Snippet): 


Ensure handwashing stations with potable 
water, soap, and a method to dry hands are 
available to workers. Additionally, ensure hand 
sanitizer with at least 60% alcohol is available.



Ensure potable water is provided for 
drinking, personal hygiene, cooking, washing 
of goods, washing of utensils, washing of 
food preparation or processing premises, 
and rooms not directly connected with the 
production or service performed by the 
establishment (e. g. , first-aid, medical services, 
and dressing).



Source